# Exercise 3: Few-shot NER for Kyrgyz

*Sequence Labelling · version 0.1.0*

> WikiANN Kyrgyz has only 100 training sentences, so use KyrgyzNER (see the KyrgyzNER paper, https://arxiv.org/abs/2509.19109) instead: first check its released splits and licence. Train a tagger (XLM-R, or a CRF where no GPU is available) on nested random subsets of its training split (for example 50, 200, 500 and 2,000 sentences — keep only sizes that fit the released training split) with three seeds each. Reserve a fixed development subset from the training split before sampling; use it for hyperparameters and checkpoint selection. Once the recipe and subset sizes are fixed, plot their F1 on the released test set without using that curve to retune the models. Compare with zero-shot transfer from a KazNERD model (the pretrained Stanza model of the baseline, or your fine-tuned XLM-R). To do so, declare a label mapping between the 25 KazNERD and 27 KyrgyzNER types before looking at any scores, map unsupported types to `O` consistently in gold and predictions, retain all tokens, and score both systems on the same mapped test set. Document boundary differences that the type mapping leaves unresolved. Report the domain difference (TV news vs. news portal) as a separate caveat. At what training size does Kyrgyz fine-tuning overtake zero-shot transfer?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "stanza==1.11.0" sklearn-crfsuite transformers sentencepiece requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`, which the first cell uses instead). Stanza is pinned to 1.11.0 because the worked solutions use its Kazakh KazNERD checkpoint; models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`), and missing ones are downloaded. The datasets are read from commit-pinned GitHub files (KazNERD, KyrgyzNER, the Starlang Turkish NER data, TRopBank, the Turkish WordNet and UD 2.17 IMST), downloaded once into `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`) and checked against SHA-256 hashes. Taggers receive the **gold words**, so tokenisation errors do not enter the scores.

**Release check.** KyrgyzNER (commit `b76b9c9` of `github.com/Akyl-AI/KyrgyzNER`) is MIT-licensed and has a training and a test split, but no development split. The paper and README describe 27 classes; the first cell counts how many occur in the release.

**The mapping, fixed before scoring.** PER: KazNERD PERSON, KyrgyzNER PERSON. LOC: KazNERD GPE and LOCATION, KyrgyzNER LOCATION. ORG: KazNERD ORGANISATION, KyrgyzNER ORGANISATION, INSTITUTION and BUSINESS. Every other type is set to O on both sides, in gold and in predictions.

**Systems.** Zero-shot: the pretrained Stanza KazNERD model applied to Kyrgyz Cyrillic, with no conversion. Few-shot: a CRF (`sklearn-crfsuite`) with word, affix, shape and context features, trained on the three mapped types. Subsets are nested within each seed: each larger subset contains the smaller one. The recorded CRF curve uses fixed settings and the released test set descriptively. For tuned extensions, reserve development data from training before sampling, select settings there, and freeze the recipe before test evaluation. With suitable hardware, compare XLM-R fine-tuning (`code/ch09-ner/train.py`).

In [ ]:
import collections, hashlib, os, random, re, statistics
from pathlib import Path
import requests

# Pinned public releases (commit-addressed raw files) and their SHA-256 hashes.
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD.format("valid"): "29d584d1a0d9aca6d58b39eb8a93e154791d303ecff56ef93f12b3c889239bf1",
    KAZNERD.format("test"): "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER.format("KyrgyzNER_TRAIN.jsonl.txt"): "719a74a18cd99d6709c1e0592d0337aafade65aa97fe762e62f3f72a6b343a26",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()

def fetch(url, sha256=None):
    """Download a pinned file once, check its hash and return the local path."""
    sha256 = sha256 or SHA256[url]
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha256:
        raise ValueError(f"{url} differs from the pinned release")
    return path

def read_conll(path, tag_col=-1, comment="#"):
    """Sentences as (words, tags) from a column file with blank-line sentence breaks."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith(comment + " "):
            cols = line.split("\t") if "\t" in line else line.split()
            words.append(cols[0]); tags.append(cols[tag_col].strip())
    return sents

def spans(tags):
    """Strict IOB2 spans {(start, end, type)}; an I- tag that does not continue a span starts none."""
    out, start, typ = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("I-") and typ == t[2:]:
            continue
        if typ is not None:
            out.add((start, i, typ))
        start, typ = (i, t[2:]) if t.startswith("B-") else (None, None)
    return out

def bioes_to_iob2(tags):
    return ["B-" + t[2:] if t[:2] in ("S-", "B-") else "I-" + t[2:] if t[:2] in ("E-", "I-") else "O" for t in tags]

def prf(gold, pred):
    """Micro precision, recall and F1 (%) over lists of span sets."""
    g = sum(len(x) for x in gold); p = sum(len(x) for x in pred)
    c = sum(len(x & y) for x, y in zip(gold, pred))
    pr = c / p if p else 0.0; rc = c / g if g else 0.0
    return {"gold": g, "pred": p, "correct": c, "P": round(100 * pr, 2), "R": round(100 * rc, 2),
            "F1": round(100 * (2 * pr * rc / (pr + rc) if pr + rc else 0.0), 2)}

def stanza_ner(lang="kk"):
    """Stanza NER over gold words; models from STANZA_RESOURCES_DIR, downloaded only if missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "ner").exists()
    return stanza.Pipeline(lang, processors="tokenize,ner", tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def tag_words(nlp, sents, batch=2000):
    """IOB2 predictions for gold-word sentences (spaces inside a token are joined with '_')."""
    out = []
    for i in range(0, len(sents), batch):
        doc = nlp([[w.replace(" ", "_") for w in words] for words, _ in sents[i:i + batch]])
        for s, (words, _) in zip(doc.sentences, sents[i:i + batch]):
            assert len(s.tokens) == len(words)
            out.append(bioes_to_iob2([t.ner for t in s.tokens]))
    return out

In [ ]:
ky_train = read_conll(fetch(KYRGYZNER.format("KyrgyzNER_TRAIN.jsonl.txt")))
ky_test = read_conll(fetch(KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt")))
types = lambda sents: collections.Counter(t[2:] for _, g in sents for t in g if t.startswith("B-"))
print(f"KyrgyzNER release: train {len(ky_train)} sentences, test {len(ky_test)} (no development split); "
      f"{len(set(types(ky_train)) | set(types(ky_test)))} entity types occur")

# The label mapping, declared before any score is computed. Types without a counterpart become O.
KAZ_MAP = {"PERSON": "PER", "GPE": "LOC", "LOCATION": "LOC", "ORGANISATION": "ORG"}
KIR_MAP = {"PERSON": "PER", "LOCATION": "LOC", "ORGANISATION": "ORG", "INSTITUTION": "ORG", "BUSINESS": "ORG"}

def mapped(tags, table):
    out = []
    for t in tags:
        m = table.get(t[2:]) if t != "O" else None
        out.append(f"{t[0]}-{m}" if m else "O")
    return out

gold = [spans(mapped(g, KIR_MAP)) for _, g in ky_test]
print("mapped test mentions:", collections.Counter(t for s in gold for *_, t in s))

# Zero-shot: the pretrained Stanza KazNERD model on Kyrgyz Cyrillic, with the same mapping.
zs = [spans(mapped(p, KAZ_MAP)) for p in tag_words(stanza_ner("kk"), ky_test)]
print("zero-shot Stanza KazNERD -> KyrgyzNER test:", prf(gold, zs))
for typ in ("PER", "LOC", "ORG"):
    print(f"   {typ}:", prf([{x for x in s if x[2] == typ} for s in gold], [{x for x in s if x[2] == typ} for s in zs]))

In [ ]:
# Few-shot: a CRF trained on nested random subsets of the Kyrgyz training split, three seeds each.
import sklearn_crfsuite

def features(words, i):
    w = words[i]; f = {"bias": 1.0, "w": w.lower(), "title": w[:1].isupper(), "upper": w.isupper(),
                       "digit": any(c.isdigit() for c in w), "len": min(len(w), 12)}
    for k in (1, 2, 3, 4):
        f[f"suf{k}"] = w[-k:].lower(); f[f"pre{k}"] = w[:k].lower()
    for d in (-2, -1, 1, 2):
        j = i + d
        f[f"w{d}"] = words[j].lower() if 0 <= j < len(words) else "<pad>"
        f[f"t{d}"] = words[j][:1].isupper() if 0 <= j < len(words) else False
    return f

X_test = [[features(ws, i) for i in range(len(ws))] for ws, _ in ky_test]
curve = {}
for n in (50, 200, 500, 2000, len(ky_train)):
    scores = []
    for seed in (1, 2, 3):
        order = list(range(len(ky_train))); random.Random(seed).shuffle(order)
        sub = [ky_train[i] for i in order[:n]]          # nested: each subset extends the smaller one
        crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100)
        crf.fit([[features(ws, i) for i in range(len(ws))] for ws, _ in sub], [mapped(g, KIR_MAP) for _, g in sub])
        scores.append(prf(gold, [spans(p) for p in crf.predict(X_test)])["F1"])
        if n == len(ky_train):
            break                                        # the full split is the same for every seed
    curve[n] = scores
    print(f"CRF trained on {n:5} Kyrgyz sentences: F1 {statistics.mean(scores):.2f}  (seeds: {scores})")

## Worked solution

Output of `experiments/ch09_exercises/run_ch09_exercises.py` (KyrgyzNER `b76b9c9`; Stanza 1.11.0):

```
KyrgyzNER release: train 7033 sentences, test 3867 (no development split); 25 entity types occur
mapped test mentions: Counter({'PER': 1151, 'LOC': 1033, 'ORG': 907})
zero-shot Stanza KazNERD -> KyrgyzNER test: {'gold': 3091, 'pred': 2428, 'correct': 1101, 'P': 45.35, 'R': 35.62, 'F1': 39.9}
   PER: {'gold': 1151, 'pred': 1326, 'correct': 555, 'P': 41.86, 'R': 48.22, 'F1': 44.81}
   LOC: {'gold': 1033, 'pred': 649, 'correct': 398, 'P': 61.33, 'R': 38.53, 'F1': 47.32}
   ORG: {'gold': 907, 'pred': 453, 'correct': 148, 'P': 32.67, 'R': 16.32, 'F1': 21.76}
CRF trained on    50 Kyrgyz sentences: F1 27.78  (seeds: [25.88, 27.54, 29.93])
CRF trained on   200 Kyrgyz sentences: F1 38.56  (seeds: [42.07, 37.19, 36.43])
CRF trained on   500 Kyrgyz sentences: F1 47.95  (seeds: [49.76, 45.87, 48.23])
CRF trained on  2000 Kyrgyz sentences: F1 56.59  (seeds: [57.89, 56.37, 55.5])
CRF trained on  7033 Kyrgyz sentences: F1 62.57  (seeds: [62.57])
```

**The release.** The training split has 7,033 sentences and the test split 3,867 (10,900 in total, as in the paper); 25 of the 27 classes occur. The mapped test set has 3,091 mentions: 1,151 PER, 1,033 LOC and 907 ORG.

**Zero-shot.** The Kazakh model reaches F1 39.9 on Kyrgyz (P 45.4, R 35.6): 44.8 for PER, 47.3 for LOC and 21.8 for ORG. ORG has the lowest F1 among the three mapped types in this sample. Boundary policy can contribute to PER errors: KyrgyzNER includes titles and positions in PERSON spans (*… орун басары Тожибек Норов*), while KazNERD annotates POSITION separately, so a name-only prediction may fail the target dataset’s exact-span criterion. Quantifying this contribution requires an error audit.

**Few-shot.** The CRF's mean F1 over three seeds is 27.8 with 50 Kyrgyz sentences, 38.6 with 200, 48.0 with 500, 56.6 with 2,000 and 62.6 with the whole training split. Seed variation is a few points for the small subsets (36.4–42.1 at 200).

**The crossing point.** The CRF mean first exceeds the zero-shot score at the tested 500-sentence setting (48.0); its 200-sentence mean is 38.6, below 39.9, though one seed exceeds it. Changing either model could move this comparison in either direction. Domain also differs (Kazakh TV news transcripts vs. the Kyrgyz 24.kg news portal), and this design does not separate domain from language or annotation effects.

## Your turn — extension

1. Plot the learning curve (mean and range per size) with the zero-shot F1 as a horizontal line.
2. Repeat with XLM-R fine-tuning on a GPU, and with the KazNERD XLM-R model of the Hands-On as the zero-shot system.